# Clase 146 — CLIP / SigLIP embeddings multimodales

CLIP entrena texto + imagen para mapear ambos al *mismo espacio* (cosine similarity). Habilita zero-shot classification y image search.

In [ ]:
USE_ST = False
try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print('sentence_transformers disponible')
except Exception as e:
    print('ST no disponible. Fallback embeddings random tagged. Motivo:', type(e).__name__)

import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Textos + 'imágenes' sintéticas

In [ ]:
texts = ['a cat sitting on a chair', 'a dog running in a park', 'a red apple on a table', 'a blue car on a road', 'a sunset over the ocean']
image_tags = ['cat_indoor', 'dog_outdoor', 'apple_kitchen', 'car_street', 'sunset_beach']
print('pairs:', list(zip(texts, image_tags)))

## 2. Encode (CLIP real o fallback)

In [ ]:
if USE_ST:
    try:
        model = SentenceTransformer('clip-ViT-B-32')
        text_emb = model.encode(texts, normalize_embeddings=True)
        # Para CPU/sin imágenes reales, simulamos imagen como mismo modelo sobre captions
        img_emb = model.encode(texts, normalize_embeddings=True)   # idealmente sería Image
        print('CLIP embeddings:', text_emb.shape)
    except Exception:
        USE_ST = False
if not USE_ST:
    # fallback: a cada par (text, image) le damos un vector cercano (mismo seed por par)
    def fake_embed(tag):
        rng = np.random.default_rng(hash(tag) % (2**32))
        return rng.normal(0, 1, 128)
    text_emb = np.array([fake_embed(t.split()[1]) + np.random.normal(0, 0.1, 128) for t in texts])
    img_emb = np.array([fake_embed(t.split('_')[0]) + np.random.normal(0, 0.1, 128) for t in image_tags])
    text_emb /= np.linalg.norm(text_emb, axis=1, keepdims=True)
    img_emb /= np.linalg.norm(img_emb, axis=1, keepdims=True)
    print('fallback embeddings:', text_emb.shape)

## 3. Matriz de similitud texto-imagen

In [ ]:
sim = text_emb @ img_emb.T          # (n_text, n_img)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(image_tags))); ax.set_xticklabels(image_tags, rotation=45, ha='right')
ax.set_yticks(range(len(texts))); ax.set_yticklabels([t[:25] for t in texts])
plt.colorbar(im, ax=ax, label='cosine sim')
for i in range(len(texts)):
    for j in range(len(image_tags)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('text–image similarity'); plt.tight_layout(); plt.show()

## 4. Zero-shot classification

Dado una imagen `i`, su clase = `argmax_j sim(text_j, img_i)`. Sin entrenar.

In [ ]:
for i, tag in enumerate(image_tags):
    pred = sim[:, i].argmax()
    correct = '✓' if pred == i else '✗'
    print(f'{correct} image[{tag}] → predicted text: "{texts[pred]}" (sim={sim[pred, i]:.3f})')
acc = (sim.argmax(0) == np.arange(len(texts))).mean()
print(f'\nzero-shot accuracy: {acc:.2%}')

## 5. Image search: dado query text, encuentra la mejor imagen

In [ ]:
query = 'an apple'
if USE_ST:
    q_emb = model.encode([query], normalize_embeddings=True)[0]
else:
    q_emb = fake_embed('apple') / np.linalg.norm(fake_embed('apple'))
scores = img_emb @ q_emb
for i in np.argsort(-scores):
    print(f'  {image_tags[i]:18s}  sim={scores[i]:+.3f}')

## 6. SigLIP vs CLIP

| Aspecto | CLIP | SigLIP |
|---------|------|--------|
| Loss | softmax contrastive (cada batch) | sigmoid pairwise (independiente) |
| Batch size | requiere batches grandes (~32k) | escala con batches chicos |
| Negativos | implícitos en el batch | explícitos por par |
| Robustez | sensible a label noise | más estable |

```python
# CLIP loss (simplificado)
logits = (T @ I.T) * temp
loss = (cross_entropy(logits, eye) + cross_entropy(logits.T, eye)) / 2

# SigLIP loss
logits = (T @ I.T) * temp + bias
labels = 2 * eye - 1                    # +1 pares match, -1 no-match
loss = -log_sigmoid(labels * logits).mean()
```

## Conclusiones

- CLIP (OpenAI 2021): dual-encoder + contrastive loss → embeddings alineados texto-imagen.
- Habilita zero-shot, retrieval, RAG multimodal sin fine-tuning.
- SigLIP (Google 2023) reemplaza softmax con sigmoid → mejor con batches chicos.
- Variantes: CLIP-ViT-L/14, EVA-CLIP, OpenCLIP, SigLIP-2.
- En prod: index con FAISS sobre img embeddings; query encode + ANN.